<p align="left">
  <a href="https://colab.research.google.com/github/wgomezf/ANNs/blob/main/Notebooks/MLP_sklearn.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" width="200">
  </a>
</p>

In [ ]:
#################################################
# Author: Wilfrido Gómez-Flores (Cinvestav)     #
# e-mail: wgomez@cinvestav.mx                   #
# Date:   february 2026                         #
# Subject:  Multi-layer perceptron with sklearn #
#################################################

In [ ]:
# Libraries
import numpy as np                                                    # Numerical array operations
import pandas as pd                                                   # Data manipulation/analysis
import matplotlib.pyplot as plt                                       # Data plotting/visualization
from sklearn.preprocessing import StandardScaler                      # Data preprocessing
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay  # Model evaluation
from sklearn.model_selection import train_test_split                  # Split data
from sklearn.neural_network import MLPClassifier                      # MLP model

In [ ]:
# Read CSV file with breast tumor morphological features
url='https://drive.google.com/file/d/10tilQVjF71MoVGqqM0ATZ8Q6KuwsoWZr/view?usp=drive_link'
url='https://drive.google.com/uc?id=' + url.split('/')[-2]
df = pd.read_csv(url)

# Get data arrays
arr = df.to_numpy()
# Options: 'binary' classification
#          'multiclass' classification
opt = "binary" #@param ["binary", "multiclass"]
X = arr[:, :-2]
if opt.lower() == 'binary':
    Y = arr[:, -2]
    labs = ['Benign', 'Malignant'] # Pathological classes
elif opt.lower() == 'multiclass':
    Y = arr[:, -1]
    labs =['B2', 'B3', 'B4', 'B5'] # BI-RADS categories

In [ ]:
# Split data into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2, stratify=Y)

# Normalize training data and apply same transformation to test data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# MLP hyperparameters
epochs = 1000 # Number of training epochs
hn  = 50      # Number of hidden neurons
eta = 1e-3    # Learning rate
lam = 1e-6    # Regularization parameter
mb  = 32      # Minibatch size

# MLP configuration
mlp = MLPClassifier(hidden_layer_sizes=(hn,), max_iter=epochs, activation='logistic',
                    learning_rate_init=eta, alpha=lam, batch_size=mb, solver='adam',
                    shuffle=True, early_stopping=True, validation_fraction=0.2,
                    n_iter_no_change=20, random_state=2)
# MLP training
mlp.fit(X_train, Y_train)

# Predict and evaluate
accuracy = mlp.score(X_test, Y_test)
Y_pred = mlp.predict(X_test) # Get predicted class labels

In [ ]:
# Get the loss curve
J = mlp.loss_curve_
plt.figure()
plt.plot(np.arange(0, len(J)), J)
plt.xlabel('Epochs')
plt.ylabel('Cross-entropy loss')
plt.title('Learning curve')
plt.show()
print()

print(f"Accuracy: {accuracy:.3f}")

# Confusion matrix
cm = confusion_matrix(Y_test, Y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labs)
disp.plot(colorbar=False, cmap='Blues')
plt.title('Confusion matrix')
plt.show()